- ReAct (Reasoning and Acting) is an agent architecture where an LLM interleaves reasoning steps with tool executions.
- The model generates thoughts, selects actions such as tool calls, observes the results, and iteratively continues until it can produce a final answer. 
 - LangGraph implements ReAct through a loop between an agent node and tool nodes, enabling dynamic decision-making and tool usage.

ReAct Loop

Thought
   ->
Action
   ->
Observation
   ->
Thought
   ->
Action
   ->
Observation
   ->
Answer

In [ ]:
from langchain_community.tools import WikipediaQueryRun

from langchain_community.utilities import WikipediaAPIWrapper

In [ ]:
api_wrapper_arxiv = WikipediaAPIWrapper(
    top_k_results=2,
    doc_content_char_limit=500
)

wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper_arxiv)

print(wiki_tool.invoke("Attention is all you need"))

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]="ReAct_agent"

In [ ]:
# TAVILY Search Tool
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_search_tool = TavilySearchResults()

print(tavily_search_tool.invoke("Provide me the recent SpaceX news"))

In [ ]:
### Custom functions

def add(a: int, b: int) -> int:
    """
    Adds two numbers and returns the sum.

    Parameters:
        a: First number
        b: Second number

    Returns:
        Sum of a and b
    """
    return a + b

def multiply(a: int, b: int) -> int:
    """
    Multiplies two numbers and returns the product.

    Parameters:
        a: First number
        b: Second number

    Returns:
        Product of a and b
    """
    return a * b
def divide(a: float, b: float) -> float:
    """
    Divides the first number by the second number.

    Parameters:
        a: Dividend
        b: Divisor

    Returns:
        Quotient of a divided by b

    Raises:
        ValueError: If divisor is zero
    """
    if b == 0:
        raise ValueError("Cannot divide by zero")

    return a / b    

In [ ]:
### combine all tools

tools =[add, multiply, divide,wiki_tool,tavily_search_tool ]

In [ ]:
# Intialize chat model

from langchain_groq import ChatGroq

llm_groq = ChatGroq(model="qwen/qwen3-32b")

llm_with_tools = llm_groq.bind_tools(tools)

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from pprint import pprint

llm_with_tools.invoke([HumanMessage(content="who own SpaceX")])


In [ ]:
# State Schema

from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing import Annotated
class State(TypedDict):
    messages:Annotated[list[AnyMessage], add_messages]

In [ ]:
# Node definition

def tool_calling_llm(state:State):
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

In [ ]:
### Entire Chatbot with LangGraph
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

builder = StateGraph(State)

# Create nodes
builder.add_node("llm_node", tool_calling_llm)
builder.add_node("tools",ToolNode(tools)) 

# Add edges
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", tools_condition)
builder.add_edge("tools", "llm_node")
builder.add_edge("llm_node", END)


graph_builder = builder.compile()

# View the graph
display(Image(graph_builder.get_graph().draw_mermaid_png()))




In [ ]:

messages=graph_builder.invoke({"messages":HumanMessage(content="what is the current weather in Dallas and what is the 100/3")}) # LLM decides to call the Tavily tool and divide tools

for m in messages['messages']:
    m.pretty_print()

  


In [ ]:
c 


### Agents Memory- The agent can remember previous messages/state and use them in later steps

1. Short-term memory → messages stored during one conversation ( uses reducers like add_message)
2. Long-term memory → saved across different conversations using checkpointer/store - Stored Externally (checkpointer, database, vector store, or memory store, allowing the agent to recall user preferences, past interactions, and learned facts over time)

### checkpointers -Stores graph state between runs which allows to resume the conversations.
- Langgraph uses a  thread_id to identify a conversation . when the same thread_id is used again , the preevious state is restored.


In [ ]:
# Construct the graph using Memory Saver Check Pointer

from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

builder = StateGraph(State)

# Create nodes
builder.add_node("llm_node", tool_calling_llm)
builder.add_node("tools",ToolNode(tools)) 

# Add edges
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", tools_condition)
builder.add_edge("tools", "llm_node")
builder.add_edge("llm_node", END)


In [ ]:
from langgraph.checkpoint.memory import MemorySaver
memory=MemorySaver() 

graph_memory=builder.compile(checkpointer=memory)

# View the graph
display(Image(graph_memory.get_graph().draw_mermaid_png()))

In [ ]:
# Specify the thread
config = {
    "configurable": {
        "thread_id": "1"
    }
}

result = graph_memory.invoke(
    {
        "messages": [
            HumanMessage(content="What is 12+23")
        ]
    },
    config=config
)
for m in result['messages']:
    m.pretty_print()

 

In [ ]:
result = graph_memory.invoke(
    {
        "messages": [
            HumanMessage(content="Divide it by 5")
        ]
    },
    config=config # Now, langgraph used this thread config to load the previous state and uses it to build the context
)
for m in result['messages']:
    m.pretty_print()
